# CIRR and FashionIQ Sanity Check

This notebook performs lightweight checks before full baseline evaluation:

1. Environment and path checks
2. Dataset loading and sample inspection
3. DataLoader batch sanity
4. Optional VISTA forward-pass smoke test
5. Optional tiny retrieval dry run on a small subset

Use this notebook to validate setup on the cluster before launching full experiments.

In [ ]:
from pathlib import Path
import os
import sys
import random
import numpy as np
import torch
from torch.utils.data import DataLoader


def find_repo_root(start: Path) -> Path:
    """Walk upwards until a folder containing src/ and scripts/ is found."""
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").exists() and (candidate / "scripts").exists():
            return candidate
    raise RuntimeError("Could not locate repository root from current working directory.")


repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

os.chdir(repo_root)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Repo root: {repo_root}")
print(f"Torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(torch.cuda.current_device())}")

Repo root: /nfs/home/hassan/multimodal-rag-cir
Torch version: 2.10.0+cu128
CUDA available: False


In [27]:
import importlib
import src.datasets.cirr as cirr_module
import src.datasets.fashioniq as fashioniq_module

importlib.reload(cirr_module)
importlib.reload(fashioniq_module)

from src.datasets.cirr import build_cirr_dataset
from src.datasets.fashioniq import build_fashioniq_dataset
from src.retrievers.vista_retriever import VistaBGERetriever
from src.fusion import fusion

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME_OR_PATH = "BAAI/bge-base-en-v1.5"
CHECKPOINT_PATH = repo_root / "models" / "Visualized_BGE" / "Visualized_base_en_v1.5.pth"

RUN_MODEL_SMOKE_TEST = True
RUN_MINI_RETRIEVAL = False

print(f"Device: {DEVICE}")
print(f"Checkpoint exists: {CHECKPOINT_PATH.exists()}")
print(f"Checkpoint path: {CHECKPOINT_PATH}")
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

Device: cpu
Checkpoint exists: True
Checkpoint path: /nfs/home/hassan/multimodal-rag-cir/models/Visualized_BGE/Visualized_base_en_v1.5.pth
False
0
No GPU


In [18]:
DATA_CIRR_IMAGES = repo_root / "data" / "cirr" / "images"
DATA_CIRR_ANN = repo_root / "data" / "cirr" / "annotations"
DATA_FIQ_IMAGES = repo_root / "data" / "fashioniq" / "images"
DATA_FIQ_ANN = repo_root / "data" / "fashioniq" / "annotations"
DATA_FIQ_LOGS = repo_root / "data" / "fashioniq" / "logs"

print("Resolved local paths used by dataset builders")
for p in [
    DATA_CIRR_IMAGES,
    DATA_CIRR_ANN,
    DATA_FIQ_IMAGES,
    DATA_FIQ_ANN,
    DATA_FIQ_LOGS,
    CHECKPOINT_PATH,
]:
    print(f"  {p} -> exists={p.exists()}")

if not all([DATA_CIRR_IMAGES.exists(), DATA_CIRR_ANN.exists(), DATA_FIQ_IMAGES.exists(), DATA_FIQ_ANN.exists(), DATA_FIQ_LOGS.exists()]):
    raise RuntimeError("One or more dataset paths are missing. Check symlinks and repository root.")

Resolved local paths used by dataset builders
  /nfs/home/hassan/multimodal-rag-cir/data/cirr/images -> exists=True
  /nfs/home/hassan/multimodal-rag-cir/data/cirr/annotations -> exists=True
  /nfs/home/hassan/multimodal-rag-cir/data/fashioniq/images -> exists=True
  /nfs/home/hassan/multimodal-rag-cir/data/fashioniq/annotations -> exists=True
  /nfs/home/hassan/multimodal-rag-cir/data/fashioniq/logs -> exists=True
  /nfs/home/hassan/multimodal-rag-cir/models/Visualized_BGE/Visualized_base_en_v1.5.pth -> exists=True


In [28]:
# Build datasets in both triplet and image modes.
cirr_val_triplets = build_cirr_dataset(split="val", mode="triplets")
cirr_test_triplets = build_cirr_dataset(split="test1", mode="triplets")
cirr_val_images = build_cirr_dataset(split="val", mode="images")

fashioniq_val_triplets = build_fashioniq_dataset(split="val", mode="triplets")
fashioniq_val_images = build_fashioniq_dataset(split="val", mode="images")

print("CIRR lengths")
print(f"  val triplets:  {len(cirr_val_triplets)}")
print(f"  test triplets: {len(cirr_test_triplets)}")
print(f"  val images:    {len(cirr_val_images)}")

print("\nFashionIQ lengths")
print(f"  val triplets:  {len(fashioniq_val_triplets)}")
print(f"  val images:    {len(fashioniq_val_images)}")

CIRR lengths
  val triplets:  4181
  test triplets: 4148
  val images:    2297

FashionIQ lengths
  val triplets:  5669
  val images:    15096


In [29]:
def summarize_sample(sample: dict, max_caption_chars: int = 120) -> None:
    print("keys:", sorted(sample.keys()))
    for key, value in sample.items():
        if torch.is_tensor(value):
            print(f"  {key}: tensor shape={tuple(value.shape)}, dtype={value.dtype}")
        elif isinstance(value, list):
            print(f"  {key}: list len={len(value)}")
        elif isinstance(value, str):
            shown = value if len(value) <= max_caption_chars else value[:max_caption_chars] + "..."
            print(f"  {key}: {shown}")
        else:
            print(f"  {key}: {value}")


print("CIRR val triplet sample")
summarize_sample(cirr_val_triplets[0])

print("\nCIRR test triplet sample")
summarize_sample(cirr_test_triplets[0])

print("\nFashionIQ val triplet sample")
summarize_sample(fashioniq_val_triplets[0])

CIRR val triplet sample
keys: ['attention_mask', 'caption', 'group_members', 'pair_id', 'reference', 'reference_name', 'target', 'target_name', 'transformed_caption']
  pair_id: 12060
  reference_name: dev-244-0-img0
  reference: <PIL.Image.Image image mode=RGB size=339x552 at 0x714A1CDEFB90>
  target: <PIL.Image.Image image mode=RGB size=236x314 at 0x714A1A111F70>
  target_name: dev-1028-1-img1
  transformed_caption: show three bottles of soft drink
  attention_mask: show three bottles of soft drink
  caption: show three bottles of soft drink
  group_members: list len=6

CIRR test triplet sample
keys: ['attention_mask', 'caption', 'group_members', 'pair_id', 'reference', 'reference_name', 'transformed_caption']
  pair_id: 12063
  reference_name: test1-147-1-img1
  reference: <PIL.Image.Image image mode=RGB size=393x260 at 0x714C0975CBC0>
  transformed_caption: remove all but one dog and add a woman hugging it
  attention_mask: remove all but one dog and add a woman hugging it
  captio

In [30]:
def sanity_collate(batch):
    out = {}
    for key in batch[0].keys():
        values = [sample[key] for sample in batch]
        if torch.is_tensor(values[0]):
            out[key] = torch.stack(values, dim=0)
        else:
            out[key] = values
    return out

cirr_loader = DataLoader(
    cirr_val_triplets,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=sanity_collate,
 )
fashioniq_loader = DataLoader(
    fashioniq_val_triplets,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=sanity_collate,
 )

cirr_batch = next(iter(cirr_loader))
fashioniq_batch = next(iter(fashioniq_loader))

print("CIRR batch fields")
for key, value in cirr_batch.items():
    if torch.is_tensor(value):
        print(f"  {key}: tensor {tuple(value.shape)} {value.dtype}")
    else:
        elem_type = type(value[0]).__name__ if len(value) > 0 else "empty"
        print(f"  {key}: list len={len(value)} elem_type={elem_type}")

print("\nFashionIQ batch fields")
for key, value in fashioniq_batch.items():
    if torch.is_tensor(value):
        print(f"  {key}: tensor {tuple(value.shape)} {value.dtype}")
    else:
        elem_type = type(value[0]).__name__ if len(value) > 0 else "empty"
        print(f"  {key}: list len={len(value)} elem_type={elem_type}")

CIRR batch fields
  pair_id: list len=4 elem_type=int
  reference_name: list len=4 elem_type=str
  reference: list len=4 elem_type=Image
  target: list len=4 elem_type=Image
  target_name: list len=4 elem_type=str
  transformed_caption: list len=4 elem_type=str
  attention_mask: list len=4 elem_type=str
  caption: list len=4 elem_type=str
  group_members: list len=4 elem_type=list

FashionIQ batch fields
  class: list len=4 elem_type=str
  candidate: list len=4 elem_type=Image
  candidate_name: list len=4 elem_type=str
  target: list len=4 elem_type=Image
  target_name: list len=4 elem_type=str
  transformed_caption: list len=4 elem_type=str
  attention_mask: list len=4 elem_type=str


In [31]:
retriever = None

if RUN_MODEL_SMOKE_TEST:
    if not CHECKPOINT_PATH.exists():
        raise FileNotFoundError(f"Missing checkpoint: {CHECKPOINT_PATH}")

    retriever = VistaBGERetriever.from_pretrained(
        model_name_or_path=MODEL_NAME_OR_PATH,
        checkpoint_path=str(CHECKPOINT_PATH),
    ).to(DEVICE)
    retriever.eval()

    # Build a transformed batch for model forward-pass sanity.
    cirr_model_triplets = build_cirr_dataset(
        split="val",
        mode="triplets",
        image_transform=retriever.image_processor,
        caption_transform=retriever.tokenizer,
    )
    cirr_model_loader = DataLoader(
        cirr_model_triplets,
        batch_size=4,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
    )
    model_batch = next(iter(cirr_model_loader))

    with torch.no_grad():
        image_embeds = retriever.vision(model_batch["reference"].to(DEVICE)).image_embeds
        text_embeds = retriever.text(
            input_ids=model_batch["transformed_caption"].to(DEVICE),
            attention_mask=model_batch["attention_mask"].to(DEVICE),
        ).text_embeds
        fused = fusion(image_embeds, text_embeds, fusion_type="sum")

    print("Smoke test embeddings")
    print("  image_embeds:", tuple(image_embeds.shape))
    print("  text_embeds:", tuple(text_embeds.shape))
    print("  fused:", tuple(fused.shape))
    print("  has_nan:", bool(torch.isnan(fused).any().item()))
else:
    print("RUN_MODEL_SMOKE_TEST is False -> skipping model loading and forward pass.")

Smoke test embeddings
  image_embeds: (4, 768)
  text_embeds: (4, 768)
  fused: (4, 768)
  has_nan: False


In [32]:
if RUN_MINI_RETRIEVAL:
    if retriever is None:
        raise RuntimeError("Enable RUN_MODEL_SMOKE_TEST first or load retriever manually.")

    # Tiny dry run: 32 queries, 256 index images on CIRR val.
    n_queries = 32
    n_index = 256

    q_ds = build_cirr_dataset(split="val", image_transform=retriever.image_processor, caption_transform=retriever.tokenizer, mode="triplets")
    i_ds = build_cirr_dataset(split="val", image_transform=retriever.image_processor, mode="images")

    q_loader = DataLoader(q_ds, batch_size=16, shuffle=False, num_workers=2)
    i_loader = DataLoader(i_ds, batch_size=64, shuffle=False, num_workers=2)

    index_feats, index_names = [], []
    with torch.no_grad():
        for batch in i_loader:
            image_feats = retriever.vision(batch["image"].to(DEVICE)).image_embeds
            index_feats.append(torch.nn.functional.normalize(image_feats, dim=-1).cpu())
            index_names.extend(batch["image_name"])
            if len(index_names) >= n_index:
                break

        index_feats = torch.cat(index_feats, dim=0)[:n_index]
        index_names = index_names[:n_index]

        total = 0
        hits_at_10 = 0

        for batch in q_loader:
            image_feats = retriever.vision(batch["reference"].to(DEVICE)).image_embeds
            text_feats = retriever.text(
                input_ids=batch["transformed_caption"].to(DEVICE),
                attention_mask=batch["attention_mask"].to(DEVICE),
            ).text_embeds
            query_feats = fusion(image_feats, text_feats, fusion_type="sum")
            query_feats = torch.nn.functional.normalize(query_feats, dim=-1).cpu()

            sims = query_feats @ index_feats.T
            top_idx = torch.argsort(sims, dim=1, descending=True)

            for row in range(top_idx.shape[0]):
                ref_name = batch["reference_name"][row]
                target_name = batch["target_name"][row]
                ranked = [index_names[j] for j in top_idx[row].tolist() if index_names[j] != ref_name]
                if target_name in ranked[:10]:
                    hits_at_10 += 1
                total += 1
                if total >= n_queries:
                    break
            if total >= n_queries:
                break

    r10 = 100.0 * hits_at_10 / max(total, 1)
    print(f"Mini CIRR Recall@10 on {total} queries with {len(index_names)} index images: {r10:.2f}")
else:
    print("RUN_MINI_RETRIEVAL is False -> skipping tiny retrieval dry run.")

RUN_MINI_RETRIEVAL is False -> skipping tiny retrieval dry run.
